# 04 · 进阶:权力算子(9 个动词 × 4 种权力)
主册「只调高度」只放开一个自由度。但真实的权力改的是**形态语法**:开发商把板楼拆成细针塔、
居民把大院细分成自建小屋、政府把高度向权力重心收拢。这本把这些「动词」教给你。

> 数据、角色、读法都和主册一样;变的是——**权力现在可以改几何,不只是高度**。

## 怎么执行
- 点一格,按 **Shift+Enter** 执行;或选单 Run → Run All Cells 从头跑。
- 结果与图会显示在该格下面。代码都在 `engine/` 文件夹,平时不用打开。

In [ ]:
# 让 notebook 找到 engine 里的代码(这格不用改,直接执行)
import sys, os
sys.path.insert(0, os.path.abspath("engine"))
import importlib, config; importlib.reload(config)
import common as C, plots
print("就绪 ·  当前站点 SLUG =", config.SLUG, "(", config.site_name(), ")")

In [ ]:
import operators as ops, measure as M
df = C.assign_all(C.current_buildings(config.SLUG))
recs = C.to_recs(df)            # 算子的工作单位:[{geom, h, sh, frozen}]
print("%s · %d 栋。算子词汇:" % (config.site_name(), len(recs)))
print("  ", list(ops.OPS.keys()))

## 一、9 个原子算子 = power 对 form 的"动词"
| 算子 | 做什么 | 谁常用 |
|---|---|---|
| `freeze` | 锁定不动 | state / 遗产 |
| `weight_height` | 按权重重分配高度 | 政府集权 |
| `concentrate` | 高度向权力重心收拢 | 政府集权(单峰) |
| `split_to_towers` | 大板楼拆成 k 座塔 | 开发商 |
| `slim` | footprint 缩、高度补偿(塔化) | 开发商 |
| `densify` | 抬高度 = 加 GFA | 开发商 |
| `infill` | 大体量细分成自建小单元 | 居民自建 |
| `level` | 高度趋同 | 居民 / 共享 |
| `open_ground` | 缩私有 footprint 释放共享地面 | 共享平权 |

每个都是纯函数 `recs -> recs`,参数可改、可单独演示。下面看两个。

## 二、算子图谱:单个动词的 before / after
**拆板成塔** `split_to_towers`:把大 footprint 沿长轴拆成 3 块(高度不变 → GFA 守恒)。

In [ ]:
b = ops.freeze(recs, who=["state"])                                   # 先锁定政府公共
a = ops.split_to_towers(b, target=["resident", "developer"], above_m2=1500, k=3)
plots.operator_demo(b, a, "拆板成塔 split_to_towers(k=3)")

**居民自建** `infill`:把大体量细分成自建小单元(细粒、低层、有机)。看栋数怎么暴涨。

In [ ]:
a2 = ops.infill(b, target=["resident", "developer"], cell_m2=120, min_h=6, max_h=21)
plots.operator_demo(b, a2, "居民自建 infill(cell≈120㎡)", color="h")

## 三、权力体制 = 算子的配方(`regimes.yaml`)
把几个动词**按序串起来**,就是一种权力如何重写城市。4 种体制各有一张「配方」:

In [ ]:
regs = ops.load_regimes()
for name, r in regs.items():
    print("【%s】%s" % (r["label"], r["fingerprint"]))
    print("   配方:", " → ".join(s["op"] for s in r["steps"]))

## 四、四种权力 → 四种形态(同色阶横比)
同一座城市、同一批楼,套四种权力体制。**形态差异不是换了数据,纯粹来自「谁更有权、用什么动词」。**

In [ ]:
after = {n: ops.apply_regime(recs, regs[n]) for n in regs}
labels = {n: regs[n]["label"] for n in regs}
plots.regime_compare(recs, after, labels=labels)

## 五、形态指纹(measure.py 量化)
肉眼之外,用指标坐实每种权力的「指纹」:开发商=瘦长比飙升(细针塔);政府=重心集中度翻倍(向权力重心收拢);
居民=栋数暴涨(细粒碎化);共享=高度CV 最低(均质)。

In [ ]:
rows, _ = M.compare(recs, after, config.SLUG)
plots.fingerprint_bars(rows, labels={"current": "现状", **labels})
for n, m in rows.items():
    print("  %-20s 瘦长 %.2f · 高度CV %.2f · 重心集中 %.2f · 栋数 %d" %
          (labels.get(n, n), m["slender"], m["h_cv"], m["concentration"], m["n"]))

## 六、权力的几何逻辑对基底不变,但结果在乎基底
把 `config.py` 的 `SLUG` 从 `lujiazui`(CBD 超高层)换成 `yuyuan`(里弄)或 `caoyang`(工人新村),
**重跑这本**:同一个 `developer_led` 配方,盖在里弄和盖在 CBD **不会长成一样**。
> 权力 × 在地是一场对话,不是盖章。这正是为什么我们坚持跑**真实多源数据**,而不是合成方块。

## 七、动手:复制粘贴换一个算子
算子就是 `recs -> recs` 的纯函数。下面**在 notebook 里**定义一个你自己的算子、登记、用进配方——
不必改 `engine/`。完整模板与说明见 **`算子替换指南.md`**。

In [ ]:
import shapely.affinity as aff

def twist(recs, target, deg=18):
    """我的算子:把目标楼的 footprint 旋转 deg 度(示范——你可以换成任何形态操作)。"""
    out = [dict(r) for r in recs]
    for r in out:
        if r["sh"] in target and not r.get("frozen"):
            r["geom"] = aff.rotate(r["geom"], deg, origin="centroid")
    return out

ops.register("twist", twist)                       # 登记到算子表(运行时,不改文件)
a3 = twist(b, target=["resident", "developer"], deg=25)
plots.operator_demo(b, a3, "我的算子 twist(旋转 25°)")
print("现在 OPS 里多了:", "twist" in ops.OPS)

**用进配方**:把 `twist` 写进一条 `steps`,`apply_regime` 就会按序施加它——和内置算子完全平权。
```yaml
my_regime:
  steps:
    - {op: freeze, who: [state]}
    - {op: twist, target: [resident, developer], deg: 25}
    - {op: densify, target: [resident, developer], far_gain: 1.5}
```
> 这就是这套框架的核心承诺:**权力体制 = 算子的配方,而算子可教、可改、可复制粘贴。**

## 诚实边界
这些 regime → 形态是**教学假设 / 可争论的语言**,不是经验断言或真实规划预测。
简化算子不含退线/日照/产权/参与;`informal` 无信号恒空;`danwei` 国家属性在用途数据里看不见。零 AI 推断。